# Web scraping: Falcon 9 launch records

Parses the IBM-referenced Wikipedia revision of the Falcon 9 launch list into a table.

In [1]:
import re
import pandas as pd
import requests
from bs4 import BeautifulSoup

URL = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
html = requests.get(URL, headers={"User-Agent": "Mozilla/5.0 (data science coursework)"}, timeout=60).text
soup = BeautifulSoup(html, "html.parser")
tables = soup.find_all("table", "wikitable plainrowheaders collapsible")
print(soup.title.string, "| launch tables:", len(tables))

List of Falcon 9 and Falcon Heavy launches - Wikipedia | launch tables: 9


In [2]:
def text(cell):
    for tag in cell.find_all(["sup", "br"]):
        tag.replace_with(" ")
    return " ".join(cell.get_text(" ", strip=True).split())


rows = []
for table in tables:
    for tr in table.find_all("tr"):
        th, td = tr.find("th"), tr.find_all("td", recursive=False)
        if th is None or not text(th).split(" ")[0].isdigit() or len(td) < 9:
            continue
        date_time = text(td[0]).split(",")
        rows.append({
            "Flight No.": int(text(th).split(" ")[0]),
            "Date": date_time[0].strip(),
            "Time": date_time[-1].strip(),
            "Version Booster": text(td[1]),
            "Launch site": text(td[2]).replace(" ,", ""),
            "Payload": text(td[3]),
            "Payload mass": text(td[4]),
            "Orbit": text(td[5]),
            "Customer": text(td[6]),
            "Launch outcome": text(td[7]),
            "Booster landing": text(td[8]),
        })

launches = pd.DataFrame(rows)
launches.to_csv("spacex_web_scraped.csv", index=False)
print(launches.shape)
launches.head()

(121, 11)


,Flight No.,Date,Time,Version Booster,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing
0,1,4 June 2010,18:45,F9 v1.0 B0003.1,CCAFS SLC-40,Dragon Spacecraft Qualification Unit,,LEO,SpaceX,Success,Failure (parachute)
1,2,8 December 2010,15:43,F9 v1.0 B0004.1,CCAFS SLC-40,Dragon demo flight C1 (Dragon C101),,LEO ( ISS ),NASA ( COTS ) NRO,Success,Failure (parachute)
2,3,22 May 2012,07:44,F9 v1.0 B0005.1,CCAFS SLC-40,Dragon demo flight C2+ (Dragon C102),"525 kg (1,157 lb)",LEO ( ISS ),NASA ( COTS ),Success,No attempt
3,4,8 October 2012,00:35,F9 v1.0 B0006.1,CCAFS SLC-40,SpaceX CRS-1 (Dragon C103),"4,700 kg (10,400 lb)",LEO ( ISS ),NASA ( CRS ),Success,No attempt
4,5,1 March 2013,15:10,F9 v1.0 B0007.1,CCAFS SLC-40,SpaceX CRS-2 (Dragon C104),"4,877 kg (10,752 lb)",LEO ( ISS ),NASA ( CRS ),Success,No attempt


In [3]:
launches["Booster landing"].str.split(" ").str[0].value_counts()

Booster landing
Success         80
No              22
Failure         11
Controlled       5
Uncontrolled     2
Precluded        1
Name: count, dtype: int64